In [1]:
# Cell 1 - Imports and experiment constants

from __future__ import annotations

import os
import sys
import time
import json
import csv
import math
import re
import warnings
from math import sqrt
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Any, Optional

import numpy as np

# Patch NumPy 2.x compatibility for DOcplex
if not hasattr(np, "float_"):
    np.float_ = np.float64

import matplotlib.pyplot as plt
import matplotlib.patches as patches

# ======================================================================
# EXPERIMENT CONFIGURATION
# ======================================================================

GAMMA_BOUNDARY: float = 100.0      # Boundary violation penalty weight
GAMMA_OVERLAP: float = 5000.0     # Overlap violation penalty weight
RHO: float = 0.01                  # Proximal regularization coefficient
EPSILON: float = 1e-3              # Convergence tolerance for step norm ||z_{k+1} - z_k||
MAX_DCA_ITER: int = 300            # Maximum DCA iterations (increased to 100)
CPLEX_THREADS: int = 1  # Utilize full CPU cores/threads
CPLEX_WORKMEM_MB: int = 8192       # Allocate 8GB working memory for CPLEX
ACTIVE_SET_MARGIN: float = 1.0     # Active pair margin: include pairs with Nij <= MARGIN (1.0 = adjacent/overlapping)
MONOTONE_RTOL: float = 1e-4        # Relative tolerance for monotone descent check

# Acceleration Settings (Accelerate.py integration)
USE_ACCELERATION: bool = True      # Enable Nesterov/FISTA extrapolation acceleration
ACCELERATE_WINDOW: int = 3         # Number of past steps to compare against (q=3)

# Early Stopping Settings
EARLY_STOP_PATIENCE: int = 20      # Stop if no objective improvement for 20 consecutive iterations
EARLY_STOP_RTOL: float = 1e-4      # Minimum relative improvement (0.01%) to qualify as progress

BENCHMARK: str = "adaptec2"
PROJECT_ROOT: Path = Path(r"D:\HaUI\scholarship\research\EfficientPlace-HaUI-main\EfficientPlace-HaUI-main\cplex_resolver")
if not (PROJECT_ROOT / "ISPD2005" / BENCHMARK).is_dir():
    for candidate in [Path(r"d:\HaUI\scholarship\research\cplex_resolver"), Path.cwd()]:
        if (candidate / "ISPD2005" / BENCHMARK).is_dir():
            PROJECT_ROOT = candidate
            break
GRID_SIZE: int = 32                # Default fallback, will read config.json if available

print("Cell 1: Imports and configuration loaded successfully.")
print(f"  Benchmark         : {BENCHMARK}")
print(f"  GAMMA_BOUNDARY    : {GAMMA_BOUNDARY}")
print(f"  GAMMA_OVERLAP     : {GAMMA_OVERLAP}")
print(f"  RHO               : {RHO}")
print(f"  EPSILON           : {EPSILON}")
print(f"  MAX_DCA_ITER      : {MAX_DCA_ITER}")
print(f"  ACTIVE_MARGIN     : {ACTIVE_SET_MARGIN}")
print(f"  CPLEX_THREADS     : {CPLEX_THREADS}")
print(f"  CPLEX_WORKMEM_MB  : {CPLEX_WORKMEM_MB}")
print(f"  USE_ACCELERATION  : {USE_ACCELERATION} (window q={ACCELERATE_WINDOW})")
print(f"  EARLY_STOPPING    : patience={EARLY_STOP_PATIENCE}, rtol={EARLY_STOP_RTOL}")


Cell 1: Imports and configuration loaded successfully.
  Benchmark         : adaptec2
  GAMMA_BOUNDARY    : 100.0
  GAMMA_OVERLAP     : 5000.0
  RHO               : 0.01
  EPSILON           : 0.001
  MAX_DCA_ITER      : 300
  ACTIVE_MARGIN     : 1.0
  CPLEX_THREADS     : 1
  CPLEX_WORKMEM_MB  : 8192
  USE_ACCELERATION  : True (window q=3)
  EARLY_STOPPING    : patience=20, rtol=0.0001


In [2]:
# Cell 2 - Experiment directories and structured logging

timestamp = time.strftime("%Y%m%d_%H%M%S")
acc_tag = f"_acc_q{ACCELERATE_WINDOW}" if USE_ACCELERATION else ""
exp_name = f"activeset_gb_{int(GAMMA_BOUNDARY)}_gov_{int(GAMMA_OVERLAP)}_rho_{RHO:.0e}_iter{MAX_DCA_ITER}{acc_tag}"
exp_dir = PROJECT_ROOT / "experiments" / BENCHMARK / exp_name / f"run_{timestamp}"

config_dir = exp_dir / "config"
logs_dir = exp_dir / "logs"
metrics_dir = exp_dir / "metrics"
placements_dir = exp_dir / "placements"
plots_dir = exp_dir / "plots"
cplex_models_dir = exp_dir / "cplex_models"

for d in [config_dir, logs_dir, metrics_dir, placements_dir, plots_dir, cplex_models_dir]:
    d.mkdir(parents=True, exist_ok=True)

log_file_path = logs_dir / "run.log"

def log_msg(msg: str):
    ts = time.strftime("%Y-%m-%d %H:%M:%S")
    formatted = f"[{ts}] {msg}"
    print(formatted)
    with log_file_path.open("a", encoding="utf-8") as f:
        f.write(formatted + "\n")

# Save experiment configuration
exp_config = {
    "benchmark": BENCHMARK,
    "gamma_boundary": GAMMA_BOUNDARY,
    "gamma_overlap": GAMMA_OVERLAP,
    "rho": RHO,
    "epsilon": EPSILON,
    "max_dca_iter": MAX_DCA_ITER,
    "cplex_threads": CPLEX_THREADS,
    "cplex_workmem_mb": CPLEX_WORKMEM_MB,
    "active_set_margin": ACTIVE_SET_MARGIN,
    "monotone_rtol": MONOTONE_RTOL,
    "use_acceleration": USE_ACCELERATION,
    "accelerate_window": ACCELERATE_WINDOW,
    "early_stop_patience": EARLY_STOP_PATIENCE,
    "early_stop_rtol": EARLY_STOP_RTOL,
    "project_root": str(PROJECT_ROOT),
    "experiment_root": str(exp_dir),
    "timestamp": timestamp,
}
with (config_dir / "experiment_config.json").open("w", encoding="utf-8") as f:
    json.dump(exp_config, f, indent=2)

log_msg("Experiment output directories initialized successfully.")
log_msg(f"Experiment root: {exp_dir}")


[2026-09-20 08:50:34] Experiment output directories initialized successfully.
[2026-09-20 08:50:34] Experiment root: D:\HaUI\scholarship\research\EfficientPlace-HaUI-main\EfficientPlace-HaUI-main\cplex_resolver\experiments\adaptec2\activeset_gb_100_gov_5000_rho_1e-02_iter300_acc_q3\run_20260920_085034


In [3]:
# Cell 3 - Validate CPLEX installation and license

try:
    import cplex
    cpx_instance = cplex.Cplex()
    cpx_ver = cpx_instance.get_version()
    del cpx_instance
except Exception as e:
    raise RuntimeError(f"Failed to initialize IBM ILOG CPLEX C++ engine: {e}")

try:
    import docplex
    import docplex.mp.model as cpx_mp
    from docplex.mp.model import Model
    from docplex.mp.linear import LinearExpr
except Exception as e:
    raise RuntimeError(f"Failed to import DOcplex Python modeling library: {e}")

log_msg(f"CPLEX local engine validated: version {cpx_ver}")
log_msg(f"DOcplex Python interface   : version {docplex.__version__}")

# Validate Bookshelf dataset and DRL input paths
bench_dir = PROJECT_ROOT / "ISPD2005" / BENCHMARK
if not bench_dir.is_dir():
    raise FileNotFoundError(f"Benchmark folder not found: {bench_dir}")

for suffix in [".nodes", ".nets", ".pl", ".scl"]:
    p = bench_dir / f"{BENCHMARK}{suffix}"
    if not p.is_file():
        raise FileNotFoundError(f"Required Bookshelf file missing: {p}")

drl_results_dir = PROJECT_ROOT / "results" / "adaptec2"
placement_grid_path = drl_results_dir / "placement_grid.npy"
if not placement_grid_path.is_file():
    raise FileNotFoundError(f"DRL placement_grid.npy not found: {placement_grid_path}")

# Read grid_size from config.json if present
drl_config_path = drl_results_dir / "config.json"
if drl_config_path.is_file():
    try:
        with drl_config_path.open("r", encoding="utf-8") as f:
            drl_cfg = json.load(f)
            if "grid_size" in drl_cfg:
                GRID_SIZE = int(drl_cfg["grid_size"])
                log_msg(f"Loaded GRID_SIZE={GRID_SIZE} from results/config.json")
    except Exception as e:
        log_msg(f"Warning reading results/config.json: {e}. Using default GRID_SIZE={GRID_SIZE}")

log_msg(f"Validation hoan tat: Benchmark={BENCHMARK}, GRID_SIZE={GRID_SIZE}")


[2026-09-20 08:50:35] CPLEX local engine validated: version 22.1.0.0
[2026-09-20 08:50:35] DOcplex Python interface   : version 2.23.221
[2026-09-20 08:50:35] Loaded GRID_SIZE=32 from results/config.json
[2026-09-20 08:50:35] Validation hoan tat: Benchmark=adaptec2, GRID_SIZE=32


In [4]:
# Cell 4 - Parse Bookshelf dataset into PlacementDB

@dataclass
class Macro:
    name: str
    width: float
    height: float
    is_macro: bool
    global_id: int

@dataclass
class Macro:
    name: str
    width: float
    height: float
    initial_x: float
    initial_y: float
    fixed: bool

@dataclass
class MacroNet:
    macro_ids: np.ndarray
    pin_macro_ids: np.ndarray
    pin_offsets: np.ndarray

@dataclass
class PlacementDB:
    benchmark: str
    macros: List[Macro]
    macro_name_to_id: Dict[str, int]
    nets: List[MacroNet]
    x_min: float
    y_min: float
    x_max: float
    y_max: float
    total_nodes: int
    total_nets: int

    @property
    def num_macros(self) -> int:
        return len(self.macros)

    @property
    def canvas_width(self) -> float:
        return self.x_max - self.x_min

    @property
    def canvas_height(self) -> float:
        return self.y_max - self.y_min

def parse_nodes(path: Path):
    nodes = {}
    total_nodes = None
    with path.open("r", encoding="utf-8", errors="replace") as handle:
        for raw in handle:
            line = raw.strip()
            if line.startswith("NumNodes"):
                total_nodes = int(line.split(":", 1)[1])
                continue
            if not line or line.startswith("#") or line.startswith("UCLA") or line.startswith("NumTerminals"):
                continue
            fields = line.split()
            if len(fields) < 3:
                continue
            try:
                width, height = float(fields[1]), float(fields[2])
            except ValueError:
                continue
            is_terminal = any(token.lower().startswith("terminal") for token in fields[3:])
            nodes[fields[0]] = (width, height, is_terminal)
    return nodes

def parse_pl(path: Path):
    positions = {}
    with path.open("r", encoding="utf-8", errors="replace") as handle:
        for raw in handle:
            line = raw.strip()
            if not line or line.startswith("#") or line.startswith("UCLA"):
                continue
            fields = line.split()
            if len(fields) < 3:
                continue
            try:
                positions[fields[0]] = (float(fields[1]), float(fields[2]), "/FIXED" in line.upper())
            except ValueError:
                continue
    return positions

def parse_scl(path: Path):
    x_min, y_min = float("inf"), float("inf")
    x_max, y_max = -float("inf"), -float("inf")
    coordinate, height, site_width = 0.0, 0.0, 1.0
    with path.open("r", encoding="utf-8", errors="replace") as handle:
        for raw in handle:
            line = raw.strip()
            if line.startswith("Coordinate"):
                coordinate = float(line.split(":", 1)[1])
            elif line.startswith("Height"):
                height = float(line.split(":", 1)[1])
            elif line.startswith("Sitewidth"):
                site_width = float(line.split(":", 1)[1])
            elif line.startswith("SubrowOrigin"):
                values = [float(v) for v in re.findall(r"[-+]?\d+(?:\.\d+)?", line)]
                if len(values) >= 2:
                    origin, num_sites = values[0], values[1]
                    x_min, y_min = min(x_min, origin), min(y_min, coordinate)
                    x_max = max(x_max, origin + num_sites * site_width)
                    y_max = max(y_max, coordinate + height)
    return x_min, y_min, x_max, y_max

def parse_macro_nets(path: Path, macro_name_to_id: Dict[str, int]):
    nets = []
    total_nets = 0
    with path.open("r", encoding="utf-8", errors="replace") as handle:
        iterator = iter(handle)
        for raw in iterator:
            line = raw.strip()
            if line.startswith("NumNets"):
                total_nets = int(line.split(":", 1)[1])
            if not line.startswith("NetDegree"):
                continue
            match = re.search(r"NetDegree\s*:\s*(\d+)", line)
            if not match:
                continue
            degree = int(match.group(1))
            pin_ids, offsets = [], []
            for _ in range(degree):
                try:
                    pin_line = next(iterator).strip()
                except StopIteration:
                    break
                fields = pin_line.split()
                if not fields or fields[0] not in macro_name_to_id:
                    continue
                numbers = re.findall(r"[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?", pin_line.split(":", 1)[-1])
                dx, dy = (float(numbers[-2]), float(numbers[-1])) if len(numbers) >= 2 else (0.0, 0.0)
                pin_ids.append(macro_name_to_id[fields[0]])
                offsets.append((dx, dy))
            unique_ids = list(dict.fromkeys(pin_ids))
            if len(unique_ids) >= 2:
                nets.append(MacroNet(
                    macro_ids=np.asarray(unique_ids, dtype=np.int64),
                    pin_macro_ids=np.asarray(pin_ids, dtype=np.int64),
                    pin_offsets=np.asarray(offsets, dtype=np.float64),
                ))
    return nets, total_nets

def build_db(b_dir: Path, b_name: str) -> PlacementDB:
    base = b_dir / b_name
    nodes = parse_nodes(base.with_suffix(".nodes"))
    pl = parse_pl(base.with_suffix(".pl"))
    macro_names = [name for name, (_, _, terminal) in nodes.items() if terminal]
    macros = []
    for name in macro_names:
        width, height, _ = nodes[name]
        x, y, fixed = pl.get(name, (0.0, 0.0, False))
        macros.append(Macro(name, width, height, x, y, fixed))
    name_to_id = {macro.name: index for index, macro in enumerate(macros)}
    nets, total_nets = parse_macro_nets(base.with_suffix(".nets"), name_to_id)
    x_min, y_min, x_max, y_max = parse_scl(base.with_suffix(".scl"))
    return PlacementDB(
        b_name, macros, name_to_id, nets,
        x_min, y_min, x_max, y_max, len(nodes), total_nets,
    )

DB = build_db(bench_dir, BENCHMARK)
log_msg(f"Bookshelf database built: macros={DB.num_macros}, nets={len(DB.nets)}, canvas={DB.canvas_width:.1f}x{DB.canvas_height:.1f}")


[2026-09-20 08:50:36] Bookshelf database built: macros=566, nets=4201, canvas=14054.0x14040.0


In [5]:
# Cell 5 - Load placement_grid.npy and convert to z0 paper centers

def validate_placement_grid(grid_array: np.ndarray, db: PlacementDB, g_size: int) -> np.ndarray:
    if grid_array.shape != (db.num_macros, 2):
        raise ValueError(f"placement_grid shape={grid_array.shape}, nhung DB can {(db.num_macros, 2)}")
    if not np.isfinite(grid_array).all():
        raise ValueError("placement_grid chua NaN hoac Inf.")
    if np.any(grid_array < 0) or np.any(grid_array >= g_size):
        raise ValueError(f"placement_grid nam ngoai [0, {g_size-1}].")
    rounded = np.rint(grid_array)
    if not np.allclose(grid_array, rounded, atol=1e-6):
        raise ValueError("placement_grid khong phai toa do grid nguyen.")
    return rounded.astype(np.int64)

def macro_geometry(db: PlacementDB) -> Tuple[np.ndarray, np.ndarray]:
    widths = np.asarray([m.width for m in db.macros], dtype=np.float64)
    heights = np.asarray([m.height for m in db.macros], dtype=np.float64)
    return widths, heights

def grid_to_paper_centers(grid_coords: np.ndarray, db: PlacementDB, g_size: int) -> np.ndarray:
    validated_grid = validate_placement_grid(grid_coords, db, g_size)
    cell_w = db.canvas_width / float(g_size)
    cell_h = db.canvas_height / float(g_size)
    widths, heights = macro_geometry(db)
    x_lower_left = validated_grid[:, 0].astype(np.float64) * cell_w
    y_lower_left = validated_grid[:, 1].astype(np.float64) * cell_h
    x_center = x_lower_left + widths / 2.0
    y_center = y_lower_left + heights / 2.0
    return np.column_stack([x_center, y_center]).astype(np.float64)

# Load and validate z0
raw_grid = np.load(placement_grid_path)
Z0_PAPER_CENTERS = grid_to_paper_centers(raw_grid, DB, GRID_SIZE)

log_msg(f"Loaded placement_grid.npy shape: {raw_grid.shape}")
log_msg(f"Z0 paper centers range: x=[{Z0_PAPER_CENTERS[:,0].min():.2f}, {Z0_PAPER_CENTERS[:,0].max():.2f}], y=[{Z0_PAPER_CENTERS[:,1].min():.2f}, {Z0_PAPER_CENTERS[:,1].max():.2f}]")


[2026-09-20 08:50:36] Loaded placement_grid.npy shape: (566, 2)
[2026-09-20 08:50:36] Z0 paper centers range: x=[48.00, 13992.81], y=[12.00, 11016.75]


In [6]:
# Cell 6 - Paper metrics Eq.(2)-(10) (Full evaluation across ALL N*(N-1)/2 pairs)

def paper_hpwl_eq5(centers_local: np.ndarray, db: PlacementDB) -> float:
    hpwl = 0.0
    for net in db.nets:
        pts = centers_local[net.macro_ids]
        hpwl += (pts[:, 0].max() - pts[:, 0].min()) + (pts[:, 1].max() - pts[:, 1].min())
    return float(hpwl)

def boundary_penalty_eq7(centers_local: np.ndarray, db: PlacementDB) -> np.ndarray:
    widths, heights = macro_geometry(db)
    x, y = centers_local[:, 0], centers_local[:, 1]
    W, H = db.canvas_width, db.canvas_height
    lb = (
        np.maximum(widths / 2.0 - x, 0.0)
        + np.maximum(x + widths / 2.0 - W, 0.0)
        + np.maximum(heights / 2.0 - y, 0.0)
        + np.maximum(y + heights / 2.0 - H, 0.0)
    )
    return lb.astype(np.float64)

def pair_terms_eq3_eq8(
    centers_local: np.ndarray,
    db: PlacementDB,
    pair_i: Optional[np.ndarray] = None,
    pair_j: Optional[np.ndarray] = None,
) -> Dict[str, Any]:
    widths, heights = macro_geometry(db)
    if pair_i is None or pair_j is None:
        pair_i, pair_j = np.triu_indices(db.num_macros, k=1)

    wij = (widths[pair_i] + widths[pair_j]) / 2.0
    hij = (heights[pair_i] + heights[pair_j]) / 2.0

    dx = np.abs(centers_local[pair_i, 0] - centers_local[pair_j, 0])
    dy = np.abs(centers_local[pair_i, 1] - centers_local[pair_j, 1])

    term_x = dx / wij - 1.0
    term_y = dy / hij - 1.0

    # Eq. (3): Nij = max(term_x, term_y)
    Nij = np.maximum(term_x, term_y)

    # Eq. (8): l_xy_hat = max(-Nij, 0)
    l_xy_hat = np.maximum(-Nij, 0.0)

    # Eq. (10): DC identity verification
    l_xy_dc = np.maximum(Nij, 0.0) - Nij
    dc_err = float(np.max(np.abs(l_xy_hat - l_xy_dc)))

    satisfied = Nij >= 0.0
    violating = ~satisfied

    return {
        "pair_i": pair_i,
        "pair_j": pair_j,
        "wij": wij,
        "hij": hij,
        "dx": dx,
        "dy": dy,
        "Nij": Nij,
        "l_xy_hat": l_xy_hat,
        "dc_identity_max_err": dc_err,
        "total_pairs": len(Nij),
        "satisfied_pairs": int(np.sum(satisfied)),
        "violating_pairs": int(np.sum(violating)),
        "violation_ratio": float(np.mean(violating)),
    }

def paper_metrics_eq2_to_eq8(
    centers_local: np.ndarray,
    db: PlacementDB,
    gamma_boundary: float,
    gamma_overlap: float,
) -> Dict[str, Any]:
    hpwl = paper_hpwl_eq5(centers_local, db)
    lb = boundary_penalty_eq7(centers_local, db)
    pair = pair_terms_eq3_eq8(centers_local, db)

    b_sum = float(np.sum(lb))
    w_b = gamma_boundary * b_sum

    ov_sum = float(np.sum(pair["l_xy_hat"]))
    w_ov = gamma_overlap * ov_sum

    F = hpwl + w_b + w_ov

    return {
        "eq2_hpwl_objective": hpwl,
        "eq3_total_pairs": pair["total_pairs"],
        "eq3_nonoverlap_satisfied_pairs": pair["satisfied_pairs"],
        "eq3_nonoverlap_violating_pairs": pair["violating_pairs"],
        "eq3_nonoverlap_violation_ratio": pair["violation_ratio"],
        "eq4_boundary_satisfied_macros": int(np.sum(lb == 0.0)),
        "eq4_boundary_violating_macros": int(np.sum(lb > 0.0)),
        "eq5_hpwl": hpwl,
        "eq7_boundary_penalty_sum": b_sum,
        "eq7_weighted_boundary_penalty": w_b,
        "eq8_overlap_hat_sum": ov_sum,
        "eq8_overlap_hat_max": float(np.max(pair["l_xy_hat"])),
        "eq8_weighted_overlap_penalty": w_ov,
        "eq6_penalized_objective": F,
        "eq10_dc_identity_max_abs_error": pair["dc_identity_max_err"],
        "Nij_min": float(np.min(pair["Nij"])),
        "Nij_max": float(np.max(pair["Nij"])),
    }

METRICS_BEFORE = paper_metrics_eq2_to_eq8(
    Z0_PAPER_CENTERS,
    DB,
    gamma_boundary=GAMMA_BOUNDARY,
    gamma_overlap=GAMMA_OVERLAP,
)

with (metrics_dir / "metrics_before.json").open("w", encoding="utf-8") as f:
    json.dump(METRICS_BEFORE, f, indent=2)

log_msg("DRL CHECKPOINT (z0) FULL METRICS (all 156,520 pairs):")
log_msg(f"  HPWL Eq.(5)                   : {METRICS_BEFORE['eq5_hpwl']:.2f}")
log_msg(f"  Overlap Violating Pairs Eq.(3): {METRICS_BEFORE['eq3_nonoverlap_violating_pairs']} / {METRICS_BEFORE['eq3_total_pairs']}")
log_msg(f"  Boundary Violating Macros Eq.(4): {METRICS_BEFORE['eq4_boundary_violating_macros']} / {DB.num_macros}")
log_msg(f"  Penalized Objective F(z0) Eq.(6): {METRICS_BEFORE['eq6_penalized_objective']:.6e}")


[2026-09-20 08:50:36] DRL CHECKPOINT (z0) FULL METRICS (all 156,520 pairs):
[2026-09-20 08:50:36]   HPWL Eq.(5)                   : 17944702.38
[2026-09-20 08:50:36]   Overlap Violating Pairs Eq.(3): 1174 / 159895
[2026-09-20 08:50:36]   Boundary Violating Macros Eq.(4): 8 / 566
[2026-09-20 08:50:36]   Penalized Objective F(z0) Eq.(6): 2.033397e+07


In [7]:
# Cell 7 - Active-Set selection for macro pairs

@dataclass
class ActiveSetData:
    active_idx: np.ndarray
    act_i: np.ndarray
    act_j: np.ndarray
    act_wij: np.ndarray
    act_hij: np.ndarray
    num_active: int
    total_pairs: int

def select_active_set(
    z: np.ndarray,
    db: PlacementDB,
    margin: float = ACTIVE_SET_MARGIN,
) -> ActiveSetData:
    """
    Select active macro pairs based on normalized separation Nij <= margin.
    Pairs with Nij > margin are far apart (do not overlap, zero subgradient, zero penalty).
    """
    pair = pair_terms_eq3_eq8(z, db)
    Nij = pair["Nij"]
    pair_i = pair["pair_i"]
    pair_j = pair["pair_j"]
    wij = pair["wij"]
    hij = pair["hij"]

    active_mask = Nij <= margin
    active_idx = np.where(active_mask)[0]

    return ActiveSetData(
        active_idx=active_idx,
        act_i=pair_i[active_idx],
        act_j=pair_j[active_idx],
        act_wij=wij[active_idx],
        act_hij=hij[active_idx],
        num_active=len(active_idx),
        total_pairs=len(pair_i),
    )

ACT0 = select_active_set(Z0_PAPER_CENTERS, DB, ACTIVE_SET_MARGIN)
log_msg(f"Active-Set at z0 (margin={ACTIVE_SET_MARGIN}): {ACT0.num_active} active pairs / {ACT0.total_pairs} total pairs ({ACT0.num_active/ACT0.total_pairs*100:.2f}%)")


[2026-09-20 08:50:36] Active-Set at z0 (margin=1.0): 3736 active pairs / 159895 total pairs (2.34%)


In [8]:
# Cell 8 - DC Decomposition audit on Active-Set

def dc_decomposition_audit(
    z: np.ndarray,
    db: PlacementDB,
    active_set: ActiveSetData,
    gamma_boundary: float,
    gamma_overlap: float,
    rho: float,
) -> Tuple[float, float, float]:
    """
    Audit DC decomposition G(z) - H(z) on the active pair set.
    For pairs not in active set, Nij > margin > 0 => overlap penalty = 0.
    """
    hpwl = paper_hpwl_eq5(z, db)
    lb = boundary_penalty_eq7(z, db)

    # Active pairs
    dx = np.abs(z[active_set.act_i, 0] - z[active_set.act_j, 0])
    dy = np.abs(z[active_set.act_i, 1] - z[active_set.act_j, 1])
    Nij_act = np.maximum(dx / active_set.act_wij - 1.0, dy / active_set.act_hij - 1.0)
    prox = 0.5 * rho * float(np.sum(z * z))

    G = hpwl + gamma_boundary * float(lb.sum()) + gamma_overlap * float(np.maximum(Nij_act, 0.0).sum()) + prox
    H = gamma_overlap * float(Nij_act.sum()) + prox
    return G, H, G - H

G0, H0, diff0 = dc_decomposition_audit(Z0_PAPER_CENTERS, DB, ACT0, GAMMA_BOUNDARY, GAMMA_OVERLAP, RHO)
F0 = METRICS_BEFORE["eq6_penalized_objective"]
log_msg(f"Active-Set DC Audit at z0: G={G0:.4e}, H={H0:.4e}, G-H={diff0:.4e}, F={F0:.4e}")
log_msg(f"Difference |(G-H) - F(z0)| = {abs(diff0 - F0):.6e}")


[2026-09-20 08:50:36] Active-Set DC Audit at z0: G=2.5536e+08, H=2.3503e+08, G-H=2.0334e+07, F=2.0334e+07
[2026-09-20 08:50:36] Difference |(G-H) - F(z0)| = 3.725290e-09


In [9]:
# Cell 9 - NumPy subgradient of H on Active-Set Eq.(18)-(19)

def compute_subgradient_H_activeset(
    z: np.ndarray,
    db: PlacementDB,
    active_set: ActiveSetData,
    gamma_overlap: float,
    rho: float,
) -> np.ndarray:
    """
    Compute subgradient y_k in \partial H(z_k) according to Eq. (18)-(19)
    restricted to the active pair set. Inactive pairs have zero overlap force.
    """
    act_i = active_set.act_i
    act_j = active_set.act_j
    act_wij = active_set.act_wij
    act_hij = active_set.act_hij

    dx = z[act_i, 0] - z[act_j, 0]
    dy = z[act_i, 1] - z[act_j, 1]
    ax = np.abs(dx) / act_wij
    ay = np.abs(dy) / act_hij

    x_active = (ax > ay).astype(np.float64)
    y_active = (ay > ax).astype(np.float64)
    tie = (ax == ay).astype(np.float64)

    x_weight = x_active + 0.5 * tie
    y_weight = y_active + 0.5 * tie

    eta_x = x_weight * np.sign(dx) / act_wij
    eta_y = y_weight * np.sign(dy) / act_hij

    gx_pair = gamma_overlap * eta_x
    gy_pair = gamma_overlap * eta_y

    # Proximal quadratic term contribution: rho * z
    grad = rho * z.copy()

    # Scatter additions for pair (i, j): +eta for i, -eta for j
    np.add.at(grad[:, 0], act_i, gx_pair)
    np.add.at(grad[:, 0], act_j, -gx_pair)
    np.add.at(grad[:, 1], act_i, gy_pair)
    np.add.at(grad[:, 1], act_j, -gy_pair)

    return grad.astype(np.float64)

log_msg("compute_subgradient_H_activeset() ready.")


[2026-09-20 08:50:36] compute_subgradient_H_activeset() ready.


In [10]:
# Cell 10 - CPLEX convex QP builder for Active-Set Eq.(17)

def build_activeset_subproblem_model(
    db: PlacementDB,
    active_set: ActiveSetData,
    yk: np.ndarray,
    gamma_boundary: float,
    gamma_overlap: float,
    rho: float,
    threads: int = CPLEX_THREADS,
    workmem_mb: int = CPLEX_WORKMEM_MB,
) -> Model:
    """
    Build CPLEX QP model for convex subproblem Eq. (17) on the active pair set.
    Variables:
      - Macro centers: x_i, y_i (free continuous, lb=-inf, ub=inf)
      - HPWL epigraphs: xmax_e, xmin_e, ymax_e, ymin_e
      - Boundary epigraphs: b_l, b_r, b_b, b_t (lb=0)
      - Overlap epigraphs: q_k (lb=0) ONLY for k in active_set
    """
    N = db.num_macros
    num_nets = len(db.nets)
    num_act = active_set.num_active
    widths, heights = macro_geometry(db)
    W, H = db.canvas_width, db.canvas_height

    mdl = Model(name="activeset_eq17")
    mdl.parameters.threads = threads
    mdl.parameters.solutiontype = 2  # Non-basic solution (Interior Point Barrier without crossover)
    mdl.parameters.workmem = workmem_mb
    mdl.parameters.barrier.display = 0
    mdl.parameters.parallel = 1
    inf = mdl.infinity

    # 1. Macro variables
    x = mdl.continuous_var_list(N, lb=-inf, ub=inf, name="x")
    y = mdl.continuous_var_list(N, lb=-inf, ub=inf, name="y")

    # 2. HPWL epigraphs
    xmax = mdl.continuous_var_list(num_nets, lb=-inf, ub=inf, name="xmax")
    xmin = mdl.continuous_var_list(num_nets, lb=-inf, ub=inf, name="xmin")
    ymax = mdl.continuous_var_list(num_nets, lb=-inf, ub=inf, name="ymax")
    ymin = mdl.continuous_var_list(num_nets, lb=-inf, ub=inf, name="ymin")

    # 3. Boundary epigraphs
    b_left = mdl.continuous_var_list(N, lb=0, ub=inf, name="b_l")
    b_right = mdl.continuous_var_list(N, lb=0, ub=inf, name="b_r")
    b_bottom = mdl.continuous_var_list(N, lb=0, ub=inf, name="b_b")
    b_top = mdl.continuous_var_list(N, lb=0, ub=inf, name="b_t")

    # 4. Overlap epigraphs for active pairs only
    q = mdl.continuous_var_list(num_act, lb=0, ub=inf, name="q")

    cts = []
    # HPWL constraints
    for e, net in enumerate(db.nets):
        xme, xne = xmax[e], xmin[e]
        yme, yne = ymax[e], ymin[e]
        for m_id in net.macro_ids:
            cts.append(xme - x[m_id] >= 0.0)
            cts.append(x[m_id] - xne >= 0.0)
            cts.append(yme - y[m_id] >= 0.0)
            cts.append(y[m_id] - yne >= 0.0)

    # Boundary constraints
    for i in range(N):
        wi2, hi2 = widths[i] / 2.0, heights[i] / 2.0
        cts.append(b_left[i] + x[i] >= wi2)
        cts.append(b_right[i] - x[i] >= wi2 - W)
        cts.append(b_bottom[i] + y[i] >= hi2)
        cts.append(b_top[i] - y[i] >= hi2 - H)

    # Overlap constraints for active pairs
    inv_w = 1.0 / active_set.act_wij
    inv_h = 1.0 / active_set.act_hij
    act_i = active_set.act_i
    act_j = active_set.act_j
    for k in range(num_act):
        i = int(act_i[k])
        j = int(act_j[k])
        iw, ih = float(inv_w[k]), float(inv_h[k])
        qk = q[k]
        cts.append(qk - iw * x[i] + iw * x[j] >= -1.0)
        cts.append(qk + iw * x[i] - iw * x[j] >= -1.0)
        cts.append(qk - ih * y[i] + ih * y[j] >= -1.0)
        cts.append(qk + ih * y[i] - ih * y[j] >= -1.0)

    mdl.add_constraints(cts)

    # Objective
    hpwl_term = mdl.sum((xmax[e] - xmin[e]) + (ymax[e] - ymin[e]) for e in range(num_nets))
    boundary_term = gamma_boundary * mdl.sum(b_left[i] + b_right[i] + b_bottom[i] + b_top[i] for i in range(N))
    overlap_term = gamma_overlap * mdl.sum(q)
    prox_term = 0.5 * rho * mdl.sum(x[i]**2 + y[i]**2 for i in range(N))
    lin_term = - mdl.scal_prod(x, yk[:, 0].tolist()) - mdl.scal_prod(y, yk[:, 1].tolist())

    mdl.minimize(hpwl_term + boundary_term + overlap_term + prox_term + lin_term)
    return mdl, x, y

log_msg("build_activeset_subproblem_model() ready.")


[2026-09-20 08:50:37] build_activeset_subproblem_model() ready.


In [11]:
# Cell 11 - solve_convex_subproblem_cplex_activeset()

def solve_convex_subproblem_cplex_activeset(
    db: PlacementDB,
    active_set: ActiveSetData,
    yk: np.ndarray,
    gamma_boundary: float,
    gamma_overlap: float,
    rho: float,
    iter_idx: int,
    cplex_models_folder: Path,
) -> Tuple[Optional[np.ndarray], float, float, str]:
    """
    Solve Active-Set convex subproblem Eq. (17) using IBM ILOG CPLEX.
    Returns:
        z_next: (N, 2) numpy array
        surrogate_obj: float
        solve_time: float (seconds)
        status_str: str ('optimal' or failure reason)
    """
    N = db.num_macros
    t0 = time.time()
    mdl, x_vars, y_vars = build_activeset_subproblem_model(
        db=db,
        active_set=active_set,
        yk=yk,
        gamma_boundary=gamma_boundary,
        gamma_overlap=gamma_overlap,
        rho=rho,
        threads=CPLEX_THREADS,
        workmem_mb=CPLEX_WORKMEM_MB,
    )
    build_time = time.time() - t0

    # Export first model LP for audit
    if iter_idx == 0:
        lp_path = cplex_models_folder / "iter_000_activeset.lp"
        try:
            mdl.export_as_lp(str(lp_path))
            log_msg(f"Exported active-set LP to: {lp_path}")
        except Exception as e:
            log_msg(f"Warning exporting LP: {e}")

    t_solve = time.time()
    sol = mdl.solve(log_output=False)
    solve_time = time.time() - t_solve

    status_name = mdl.solve_details.status if mdl.solve_details else "unknown"

    if sol is None:
        log_msg(f"ERROR: CPLEX returned no solution at iteration {iter_idx}. Status: {status_name}")
        failed_lp = cplex_models_folder / f"failed_iter_{iter_idx:03d}.lp"
        try:
            mdl.export_as_lp(str(failed_lp))
            log_msg(f"Exported failed model to: {failed_lp}")
        except Exception:
            pass
        return None, float("nan"), solve_time, "cplex_solver_failed"

    z_next = np.empty((N, 2), dtype=np.float64)
    z_next[:, 0] = [sol.get_value(x_vars[i]) for i in range(N)]
    z_next[:, 1] = [sol.get_value(y_vars[i]) for i in range(N)]

    if not np.isfinite(z_next).all():
        log_msg(f"ERROR: Non-finite values in CPLEX solution at iteration {iter_idx}")
        return None, float("nan"), solve_time, "cplex_solver_failed"

    surrogate_val = float(sol.objective_value)
    return z_next, surrogate_val, solve_time + build_time, "optimal"

log_msg("solve_convex_subproblem_cplex_activeset() ready.")


[2026-09-20 08:50:37] solve_convex_subproblem_cplex_activeset() ready.


In [12]:
# Cell 12 - Proximal DCA Algorithm loop with Active-Set CPLEX and Accelerate.py

class Accelerate(object):
    """
    Nesterov/FISTA extrapolation accelerator for DCA (Accelerate.py integration).
    Compares candidate objective against the last q steps. If candidate improves
    upon any of the last q values, it is accepted; otherwise safely falls back.
    """
    def __init__(self, acc: bool = False):
        self.iter = 0
        self.Acc = acc
        self.zu = np.array([])
        self.zv = np.array([])
        self.zz = np.array([])
        self.F = np.array([0.0])
        self.t = (1.0 + sqrt(5.0)) / 2.0

    def push(self, uk=None, ukplus=None, vk=None, vkplus=None, zk=None, zkplus=None, nbVar=1):
        if self.Acc:
            tkplus = (1.0 + sqrt(1.0 + 4.0 * self.t * self.t)) / 2.0
            cof = (self.t - 1.0) / tkplus
            self.t = tkplus
            if uk is not None and ukplus is not None:
                self.zu = ukplus + cof * (ukplus - uk)
            if vk is not None and vkplus is not None:
                self.zv = vkplus + cof * (vkplus - vk)
            if zk is not None and zkplus is not None:
                self.zz = zkplus + cof * (zkplus - zk)

    def record_fallback(self, F_val: float):
        """Record function value when acceleration candidate is rejected."""
        self.F = np.append(self.F, float(F_val))

    def acceptz(self, Fz: float, q: int = 3) -> bool:
        if self.Acc:
            if len(self.F) == 1:
                self.F = np.append(self.F, float(Fz))
                return True
            q_eff = min(q, len(self.F))
            accept = False
            for i in range(q_eff):
                if Fz <= self.F[len(self.F) - i - 1]:
                    accept = True
                    break
            if accept:
                self.F = np.append(self.F, float(Fz))
            return accept
        return False

@dataclass
class DCAResult:
    z_star: np.ndarray
    history: List[Dict[str, Any]]
    final_status: str
    total_time_sec: float
    converged: bool
    iterations_run: int

def run_proximal_dca_activeset(
    z0: np.ndarray,
    db: PlacementDB,
    gamma_boundary: float,
    gamma_overlap: float,
    rho: float,
    epsilon: float,
    max_dca_iter: int,
    active_margin: float,
    monotone_rtol: float,
    cplex_models_folder: Path,
    use_acceleration: bool = USE_ACCELERATION,
    accelerate_window: int = ACCELERATE_WINDOW,
    early_stop_patience: int = EARLY_STOP_PATIENCE,
    early_stop_rtol: float = EARLY_STOP_RTOL,
) -> DCAResult:
    t_total_start = time.time()
    zk = z0.copy().astype(np.float64)
    history = []
    final_status = "unknown"

    # Evaluate full F(z0) across ALL pairs
    m_curr = paper_metrics_eq2_to_eq8(zk, db, gamma_boundary=gamma_boundary, gamma_overlap=gamma_overlap)
    F_curr = m_curr["eq6_penalized_objective"]

    # Initialize Accelerate
    acc = Accelerate(acc=use_acceleration)
    acc.F[0] = F_curr

    # Early Stopping state
    best_F = F_curr
    best_z = zk.copy()
    no_improve_count = 0

    log_msg("=" * 80)
    log_msg(f"STARTING PROXIMAL DCA (ACTIVE-SET CPLEX): max_iter={max_dca_iter}, eps={epsilon}, margin={active_margin}, threads={CPLEX_THREADS}, accelerate={use_acceleration} (q={accelerate_window}), early_stop_patience={early_stop_patience}")
    log_msg(f"Iter   0 (Start) | F={F_curr:.6e} | HPWL={m_curr['eq5_hpwl']:.2f} | OverlapPairs={m_curr['eq3_nonoverlap_violating_pairs']} | BoundMacros={m_curr['eq4_boundary_violating_macros']}")
    log_msg("=" * 80)

    for k in range(max_dca_iter):
        t_iter_start = time.time()

        # Step 1: Dynamically determine active pairs at current placement zk
        active_set = select_active_set(zk, db, margin=active_margin)

        # Step 2: Compute subgradient y_k in \partial H(z_k) on active set
        yk = compute_subgradient_H_activeset(
            zk, db, active_set=active_set, gamma_overlap=gamma_overlap, rho=rho
        )

        # Step 3: Solve convex subproblem Eq. (17) using CPLEX
        z_next_qp, surrogate_obj, cplex_time, status = solve_convex_subproblem_cplex_activeset(
            db=db,
            active_set=active_set,
            yk=yk,
            gamma_boundary=gamma_boundary,
            gamma_overlap=gamma_overlap,
            rho=rho,
            iter_idx=k,
            cplex_models_folder=cplex_models_folder,
        )

        if status != "optimal" or z_next_qp is None:
            final_status = "cplex_solver_failed"
            log_msg(f"DCA Aborted at iter {k}: CPLEX solver failed.")
            break

        # Step 4: Standard QP metrics evaluation
        m_qp = paper_metrics_eq2_to_eq8(z_next_qp, db, gamma_boundary=gamma_boundary, gamma_overlap=gamma_overlap)
        F_qp = m_qp["eq6_penalized_objective"]

        # Step 5: Acceleration step (Nesterov momentum extrapolation + q=3 acceptance check)
        was_accelerated = False
        if acc.Acc:
            acc.push(vk=zk, vkplus=z_next_qp, nbVar=1)
            m_zv = paper_metrics_eq2_to_eq8(acc.zv, db, gamma_boundary=gamma_boundary, gamma_overlap=gamma_overlap)
            Fz = m_zv["eq6_penalized_objective"]

            if acc.acceptz(Fz, q=accelerate_window):
                z_next = acc.zv
                m_next = m_zv
                F_next = Fz
                was_accelerated = True
            else:
                z_next = z_next_qp
                m_next = m_qp
                F_next = F_qp
                acc.record_fallback(F_next)
        else:
            z_next = z_next_qp
            m_next = m_qp
            F_next = F_qp

        # Monotonicity audit: F_{k+1} <= F_k + tol
        allowable_tol = max(monotone_rtol * abs(F_curr), 1e-4)
        if F_next > F_curr + allowable_tol:
            log_msg(f"WARNING: Monotonicity threshold exceeded at iter {k}! F_before={F_curr:.6e}, F_after={F_next:.6e}, diff={F_next - F_curr:.6e}")

        # Step 6: Step norm
        step_norm = float(np.linalg.norm(z_next - zk))
        iter_runtime = time.time() - t_iter_start

        # Record history row
        row = {
            "iteration": k,
            "F_before": F_curr,
            "F_after": F_next,
            "HPWL_after": m_next["eq5_hpwl"],
            "boundary_penalty_sum": m_next["eq7_boundary_penalty_sum"],
            "weighted_boundary_penalty": m_next["eq7_weighted_boundary_penalty"],
            "overlap_hat_sum": m_next["eq8_overlap_hat_sum"],
            "weighted_overlap_penalty": m_next["eq8_weighted_overlap_penalty"],
            "overlap_violating_pairs": m_next["eq3_nonoverlap_violating_pairs"],
            "boundary_violating_macros": m_next["eq4_boundary_violating_macros"],
            "active_pairs": active_set.num_active,
            "step_norm": step_norm,
            "was_accelerated": was_accelerated,
            "cplex_surrogate_objective": surrogate_obj,
            "cplex_solve_time_sec": cplex_time,
            "cplex_status": status,
            "total_runtime_sec": iter_runtime,
        }
        history.append(row)

        acc_flag = "Acc=Y" if was_accelerated else "Acc=N"
        log_msg(
            f"Iter {k+1:3d} | F={F_next:.6e} | HPWL={m_next['eq5_hpwl']:.2f} | "
            f"OvPairs={m_next['eq3_nonoverlap_violating_pairs']} | BndMac={m_next['eq4_boundary_violating_macros']} | "
            f"ActPairs={active_set.num_active} | ||dz||={step_norm:.2f} | {acc_flag} | Time={iter_runtime:.2f}s"
        )

        zk = z_next
        F_curr = F_next

        # Early Stopping criterion: check if F_next improves upon best_F
        if F_next < best_F * (1.0 - early_stop_rtol):
            best_F = F_next
            best_z = z_next.copy()
            no_improve_count = 0
        else:
            no_improve_count += 1

        if early_stop_patience > 0 and no_improve_count >= early_stop_patience:
            final_status = f"early_stopped_stagnation_{early_stop_patience}_iters"
            log_msg(f"Early stopping triggered at iter {k+1}: no improvement in {early_stop_patience} consecutive iterations (best_F={best_F:.6e})")
            zk = best_z
            break

        # Step 7: Stopping criterion
        if step_norm <= epsilon:
            final_status = "converged_by_epsilon"
            log_msg(f"PDCA converged at iter {k+1}: step_norm ({step_norm:.4e}) <= EPSILON ({epsilon})")
            break

    if final_status == "unknown":
        final_status = "max_iterations_reached"
        log_msg(f"PDCA reached max iterations ({max_dca_iter}).")

    total_time = time.time() - t_total_start
    log_msg("=" * 80)
    log_msg(f"PDCA FINISHED: status={final_status}, total_time={total_time:.2f}s, iterations={len(history)}")
    log_msg("=" * 80)

    return DCAResult(
        z_star=zk,
        history=history,
        final_status=final_status,
        total_time_sec=total_time,
        converged=(final_status in ("converged_by_epsilon", f"early_stopped_stagnation_{early_stop_patience}_iters")),
        iterations_run=len(history),
    )

log_msg("run_proximal_dca_activeset() ready.")


[2026-09-20 08:50:37] run_proximal_dca_activeset() ready.


In [13]:
# Cell 13 - Run Proximal DCA from z0

DCA_RES = run_proximal_dca_activeset(
    z0=Z0_PAPER_CENTERS,
    db=DB,
    gamma_boundary=GAMMA_BOUNDARY,
    gamma_overlap=GAMMA_OVERLAP,
    rho=RHO,
    epsilon=EPSILON,
    max_dca_iter=MAX_DCA_ITER,
    active_margin=ACTIVE_SET_MARGIN,
    monotone_rtol=MONOTONE_RTOL,
    cplex_models_folder=cplex_models_dir,
    use_acceleration=USE_ACCELERATION,
    accelerate_window=ACCELERATE_WINDOW,
    early_stop_patience=EARLY_STOP_PATIENCE,
    early_stop_rtol=EARLY_STOP_RTOL,
)

Z_STAR = DCA_RES.z_star


[2026-09-20 08:50:37] ================================================================================
[2026-09-20 08:50:37] STARTING PROXIMAL DCA (ACTIVE-SET CPLEX): max_iter=300, eps=0.001, margin=1.0, threads=1, accelerate=True (q=3), early_stop_patience=20
[2026-09-20 08:50:37] Iter   0 (Start) | F=2.033397e+07 | HPWL=17944702.38 | OverlapPairs=1174 | BoundMacros=8
[2026-09-20 08:50:37] ================================================================================
[2026-09-20 08:50:39] Exported active-set LP to: D:\HaUI\scholarship\research\EfficientPlace-HaUI-main\EfficientPlace-HaUI-main\cplex_resolver\experiments\adaptec2\activeset_gb_100_gov_5000_rho_1e-02_iter300_acc_q3\run_20260920_085034\cplex_models\iter_000_activeset.lp
[2026-09-20 08:50:47] Iter   1 | F=1.158307e+07 | HPWL=8074905.82 | OvPairs=2098 | BndMac=3 | ActPairs=3736 | ||dz||=25484.94 | Acc=Y | Time=10.42s
[2026-09-20 08:51:01] Iter   2 | F=9.518360e+06 | HPWL=6850786.66 | OvPairs=1751 | BndMac=4 | ActPairs=6882

In [14]:
# Cell 14 - Save final placement and write Bookshelf .pl

def paper_centers_to_global_lower_left(centers_local: np.ndarray, db: PlacementDB) -> np.ndarray:
    widths, heights = macro_geometry(db)
    x_ll = centers_local[:, 0] - widths / 2.0
    y_ll = centers_local[:, 1] - heights / 2.0
    return np.column_stack([x_ll, y_ll]).astype(np.float64)

Z_STAR_LOWER_LEFT = paper_centers_to_global_lower_left(Z_STAR, DB)

# Save numpy arrays
np.save(placements_dir / "z_star_paper_centers.npy", Z_STAR)
np.save(placements_dir / "z_star_lower_left_global.npy", Z_STAR_LOWER_LEFT)

# Write output Bookshelf .pl file
def write_bookshelf_pl_from_paper_centers(
    centers_local: np.ndarray,
    db: PlacementDB,
    source_pl: Path,
    dest_pl: Path,
):
    lower_left = paper_centers_to_global_lower_left(centers_local, db)
    coord_by_name = {
        macro.name: lower_left[idx]
        for idx, macro in enumerate(db.macros)
    }

    out_lines = []
    with source_pl.open("r", encoding="utf-8", errors="replace") as f:
        for raw in f:
            fields = raw.strip().split()
            if fields and fields[0] in coord_by_name and len(fields) >= 3:
                x_val, y_val = coord_by_name[fields[0]]
                fields[1] = f"{x_val:.6f}"
                fields[2] = f"{y_val:.6f}"
                out_lines.append("\t".join(fields))
            else:
                out_lines.append(raw.rstrip("\r\n"))

    with dest_pl.open("w", encoding="utf-8") as f:
        f.write("\n".join(out_lines) + "\n")

final_pl_path = placements_dir / f"{BENCHMARK}.drl_pdca_cplex_activeset.pl"
write_bookshelf_pl_from_paper_centers(
    centers_local=Z_STAR,
    db=DB,
    source_pl=bench_dir / f"{BENCHMARK}.pl",
    dest_pl=final_pl_path,
)
log_msg(f"Saved final Bookshelf placement to: {final_pl_path}")


[2026-09-20 21:37:35] Saved final Bookshelf placement to: D:\HaUI\scholarship\research\EfficientPlace-HaUI-main\EfficientPlace-HaUI-main\cplex_resolver\experiments\adaptec2\activeset_gb_100_gov_5000_rho_1e-02_iter300_acc_q3\run_20260920_085034\placements\adaptec2.drl_pdca_cplex_activeset.pl


In [15]:
# Cell 15 - Save metrics/history and print comparison table

METRICS_AFTER = paper_metrics_eq2_to_eq8(
    Z_STAR,
    DB,
    gamma_boundary=GAMMA_BOUNDARY,
    gamma_overlap=GAMMA_OVERLAP,
)

with (metrics_dir / "metrics_after.json").open("w", encoding="utf-8") as f:
    json.dump(METRICS_AFTER, f, indent=2)

# Save dca_history.csv
csv_path = metrics_dir / "dca_history.csv"
fieldnames = [
    "iteration", "F_before", "F_after", "HPWL_after",
    "boundary_penalty_sum", "weighted_boundary_penalty",
    "overlap_hat_sum", "weighted_overlap_penalty",
    "overlap_violating_pairs", "boundary_violating_macros",
    "active_pairs", "step_norm", "was_accelerated", "cplex_surrogate_objective",
    "cplex_solve_time_sec", "cplex_status", "total_runtime_sec",
]
with csv_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for row in DCA_RES.history:
        writer.writerow(row)
log_msg(f"Saved DCA history to: {csv_path}")

acc_count = sum(1 for r in DCA_RES.history if r.get("was_accelerated", False))
log_msg(f"Accelerated steps accepted: {acc_count} / {DCA_RES.iterations_run}")

# Comparison Table
header = f"{'Metric':<35} | {'Before (z0 DRL)':<20} | {'After (z* DCA)':<20} | {'Change':<15}"
sep = "-" * len(header)
print("\n" + sep)
print(header)
print(sep)

comp_rows = [
    ("Penalized Objective F Eq.(6)", METRICS_BEFORE["eq6_penalized_objective"], METRICS_AFTER["eq6_penalized_objective"], True),
    ("HPWL Eq.(5)", METRICS_BEFORE["eq5_hpwl"], METRICS_AFTER["eq5_hpwl"], True),
    ("Overlap Violating Pairs Eq.(3)", METRICS_BEFORE["eq3_nonoverlap_violating_pairs"], METRICS_AFTER["eq3_nonoverlap_violating_pairs"], False),
    ("Overlap Penalty Sum Eq.(8)", METRICS_BEFORE["eq8_overlap_hat_sum"], METRICS_AFTER["eq8_overlap_hat_sum"], True),
    ("Boundary Violating Macros Eq.(4)", METRICS_BEFORE["eq4_boundary_violating_macros"], METRICS_AFTER["eq4_boundary_violating_macros"], False),
    ("Boundary Penalty Sum Eq.(7)", METRICS_BEFORE["eq7_boundary_penalty_sum"], METRICS_AFTER["eq7_boundary_penalty_sum"], True),
]

for name, v_before, v_after, is_float in comp_rows:
    if is_float:
        diff_str = f"{v_after - v_before:+.2e}"
        print(f"{name:<35} | {v_before:<20.4e} | {v_after:<20.4e} | {diff_str:<15}")
    else:
        diff_str = f"{v_after - v_before:+d}"
        print(f"{name:<35} | {v_before:<20d} | {v_after:<20d} | {diff_str:<15}")
print(sep + "\n")


[2026-09-20 21:37:35] Saved DCA history to: D:\HaUI\scholarship\research\EfficientPlace-HaUI-main\EfficientPlace-HaUI-main\cplex_resolver\experiments\adaptec2\activeset_gb_100_gov_5000_rho_1e-02_iter300_acc_q3\run_20260920_085034\metrics\dca_history.csv
[2026-09-20 21:37:35] Accelerated steps accepted: 226 / 273

---------------------------------------------------------------------------------------------------
Metric                              | Before (z0 DRL)      | After (z* DCA)       | Change         
---------------------------------------------------------------------------------------------------
Penalized Objective F Eq.(6)        | 2.0334e+07           | 4.1494e+06           | -1.62e+07      
HPWL Eq.(5)                         | 1.7945e+07           | 3.1594e+06           | -1.48e+07      
Overlap Violating Pairs Eq.(3)      | 1174                 | 790                  | -384           
Overlap Penalty Sum Eq.(8)          | 3.9914e+02           | 1.9799e+02           | -

In [16]:
# Cell 16 - Generate and save comparison and convergence plots

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

W, H = DB.canvas_width, DB.canvas_height
widths, heights = macro_geometry(DB)

ll_before = paper_centers_to_global_lower_left(Z0_PAPER_CENTERS, DB)
ll_after = paper_centers_to_global_lower_left(Z_STAR, DB)

# Plot Before
ax0 = axes[0]
ax0.set_xlim(0, W)
ax0.set_ylim(0, H)
ax0.set_aspect("equal")
ax0.set_title(f"DRL Checkpoint (z0)\nHPWL={METRICS_BEFORE['eq5_hpwl']:.0f}, OverlapPairs={METRICS_BEFORE['eq3_nonoverlap_violating_pairs']}")
for i in range(DB.num_macros):
    rect = patches.Rectangle(
        (ll_before[i, 0], ll_before[i, 1]),
        widths[i], heights[i],
        linewidth=0.5, edgecolor="blue", facecolor="cyan", alpha=0.3
    )
    ax0.add_patch(rect)

# Plot After
ax1 = axes[1]
ax1.set_xlim(0, W)
ax1.set_ylim(0, H)
ax1.set_aspect("equal")
ax1.set_title(f"After CPLEX Active-Set PDCA (z*)\nHPWL={METRICS_AFTER['eq5_hpwl']:.0f}, OverlapPairs={METRICS_AFTER['eq3_nonoverlap_violating_pairs']}")
for i in range(DB.num_macros):
    rect = patches.Rectangle(
        (ll_after[i, 0], ll_after[i, 1]),
        widths[i], heights[i],
        linewidth=0.5, edgecolor="darkgreen", facecolor="lightgreen", alpha=0.3
    )
    ax1.add_patch(rect)

plt.tight_layout()
plot_placement_path = plots_dir / "checkpoint_vs_pdca.png"
plt.savefig(plot_placement_path, dpi=200)
plt.close()
log_msg(f"Saved plot: {plot_placement_path}")

# Convergence Curves
if DCA_RES.history:
    iters = [row["iteration"] + 1 for row in DCA_RES.history]
    f_vals = [row["F_after"] for row in DCA_RES.history]
    hpwl_vals = [row["HPWL_after"] for row in DCA_RES.history]
    ov_vals = [row["overlap_violating_pairs"] for row in DCA_RES.history]
    step_norms = [row["step_norm"] for row in DCA_RES.history]

    fig, axs = plt.subplots(2, 2, figsize=(14, 10))

    axs[0, 0].plot(iters, f_vals, "b-o")
    axs[0, 0].set_title("Penalized Objective F(z)")
    axs[0, 0].set_xlabel("Iteration")
    axs[0, 0].grid(True)

    axs[0, 1].plot(iters, hpwl_vals, "g-s")
    axs[0, 1].set_title("HPWL Eq.(5)")
    axs[0, 1].set_xlabel("Iteration")
    axs[0, 1].grid(True)

    axs[1, 0].plot(iters, ov_vals, "r-^")
    axs[1, 0].set_title("Overlap Violating Pairs Eq.(3)")
    axs[1, 0].set_xlabel("Iteration")
    axs[1, 0].grid(True)

    axs[1, 1].plot(iters, step_norms, "m-d")
    axs[1, 1].set_title("Step Norm ||z_{k+1} - z_k||")
    axs[1, 1].set_xlabel("Iteration")
    axs[1, 1].grid(True)

    plt.tight_layout()
    plot_conv_path = plots_dir / "convergence_curves.png"
    plt.savefig(plot_conv_path, dpi=200)
    plt.close()
    log_msg(f"Saved plot: {plot_conv_path}")

log_msg("All plots generated and saved successfully.")


[2026-09-20 21:37:37] Saved plot: D:\HaUI\scholarship\research\EfficientPlace-HaUI-main\EfficientPlace-HaUI-main\cplex_resolver\experiments\adaptec2\activeset_gb_100_gov_5000_rho_1e-02_iter300_acc_q3\run_20260920_085034\plots\checkpoint_vs_pdca.png
[2026-09-20 21:37:38] Saved plot: D:\HaUI\scholarship\research\EfficientPlace-HaUI-main\EfficientPlace-HaUI-main\cplex_resolver\experiments\adaptec2\activeset_gb_100_gov_5000_rho_1e-02_iter300_acc_q3\run_20260920_085034\plots\convergence_curves.png
[2026-09-20 21:37:38] All plots generated and saved successfully.


In [17]:
# Cell 17 - Final summary report and validation checks

acc_count = sum(1 for r in DCA_RES.history if r.get("was_accelerated", False))

log_msg("=" * 80)
log_msg("DCA (ACTIVE-SET CPLEX) EXPERIMENT SUMMARY")
log_msg("=" * 80)
log_msg(f"Benchmark                  : {BENCHMARK}")
log_msg(f"DCA Convergence Status     : {DCA_RES.final_status}")
log_msg(f"Total Iterations Run       : {DCA_RES.iterations_run}")
log_msg(f"Accelerated Steps Accepted : {acc_count} / {DCA_RES.iterations_run}")
log_msg(f"Total Runtime              : {DCA_RES.total_time_sec:.2f}s")
log_msg(f"Average Time per Iteration : {DCA_RES.total_time_sec / max(DCA_RES.iterations_run, 1):.2f}s")
log_msg(f"Initial Overlap Pairs (z0) : {METRICS_BEFORE['eq3_nonoverlap_violating_pairs']}")
log_msg(f"Final Overlap Pairs (z*)   : {METRICS_AFTER['eq3_nonoverlap_violating_pairs']}")
log_msg(f"Initial HPWL (z0)          : {METRICS_BEFORE['eq5_hpwl']:.2f}")
log_msg(f"Final HPWL (z*)            : {METRICS_AFTER['eq5_hpwl']:.2f}")
log_msg(f"Initial Objective F(z0)    : {METRICS_BEFORE['eq6_penalized_objective']:.6e}")
log_msg(f"Final Objective F(z*)      : {METRICS_AFTER['eq6_penalized_objective']:.6e}")
log_msg(f"Output Placement .pl       : {final_pl_path}")
log_msg(f"DCA History CSV            : {csv_path}")
log_msg("=" * 80)

# Final assertions
assert final_pl_path.is_file(), "Final .pl file must exist"
assert csv_path.is_file(), "DCA history CSV must exist"
assert (metrics_dir / "metrics_before.json").is_file(), "metrics_before.json must exist"
assert (metrics_dir / "metrics_after.json").is_file(), "metrics_after.json must exist"
print("SUCCESS: All experiment outputs verified and saved properly.")


[2026-09-20 21:37:38] ================================================================================
[2026-09-20 21:37:38] DCA (ACTIVE-SET CPLEX) EXPERIMENT SUMMARY
[2026-09-20 21:37:38] ================================================================================
[2026-09-20 21:37:38] Benchmark                  : adaptec2
[2026-09-20 21:37:38] DCA Convergence Status     : early_stopped_stagnation_20_iters
[2026-09-20 21:37:38] Total Iterations Run       : 273
[2026-09-20 21:37:38] Accelerated Steps Accepted : 226 / 273
[2026-09-20 21:37:38] Total Runtime              : 46017.95s
[2026-09-20 21:37:38] Average Time per Iteration : 168.56s
[2026-09-20 21:37:38] Initial Overlap Pairs (z0) : 1174
[2026-09-20 21:37:38] Final Overlap Pairs (z*)   : 790
[2026-09-20 21:37:38] Initial HPWL (z0)          : 17944702.38
[2026-09-20 21:37:38] Final HPWL (z*)            : 3159405.41
[2026-09-20 21:37:38] Initial Objective F(z0)    : 2.033397e+07
[2026-09-20 21:37:38] Final Objective F(z*)      